In [ ]:
import pandas as pd
import os
DATA_PATH = "/content/drive/MyDrive/LG 헬로비전 부트캠프/원본 데이터셋/데이터셋/two_tower"
user_df = pd.read_parquet(os.path.join(DATA_PATH, "user_sequence_df.parquet"))
item_df = pd.read_parquet(os.path.join(DATA_PATH, "item_df_tfidf_emotion.parquet"))

In [ ]:
user_df

,user_id,watched_item_ids,watch_weights,hour_seq,age
0,0,"[CCS000000000000001, T20211209079813, T2021102...","[0.07191027613624934, 0.9932206034177273, 0.38...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",50
1,1,[T20201119066686],[0.9648576508871625],[20],40
2,4,"[T20220819088029, T20220819088029, T2022081908...","[0.9930650508401643, 0.9837723299173877, 0.993...","[22, 19, 20]",40
3,5,"[CCS000000000000001, CCS000000000000001, CCS00...","[0.052077553027645274, 0.052077553027645274, 0...","[19, 19, 19, 13, 21, 16, 16, 16, 19, 19, 19, 1...",70
4,6,"[CCS000000000000001, T20211029078323, T2021102...","[0.07191027613624934, 0.0020325196247731325, 0...","[2, 14, 14, 14, 3, 5, 5, 0, 1, 10, 12, 11, 4, ...",60
...,...,...,...,...,...
424405,612784,"[T20220718086789, T20221216091845, T2021061107...","[0.03159028009767495, 0.9921370984621719, 0.64...","[6, 17, 15, 5, 18, 9, 16, 11, 15, 20, 6, 7, 14...",60
424406,612785,"[T20230120092954, T20230120092954, T2023012009...","[0.9116525705421166, 0.7620913092991473, 0.624...","[20, 21, 21, 15, 15, 16, 17]",50
424407,612786,"[T20220718086789, T20220718086789, T2022071808...","[0.5821420000307191, 0.1971855755478017, 0.215...","[15, 15, 15, 15, 17, 23, 22, 23, 23, 23, 23, 2...",50
424408,612788,"[T20230322094961, T20220617085786]","[0.11480845247503935, 0.992229052243992]","[21, 7]",50


In [ ]:
item_df

,0,1,2,3,4,5,6,7,8,9,...,emotion_anger,emotion_anticipation,emotion_disgust,emotion_fear,emotion_joy,emotion_negative,emotion_positive,emotion_sadness,emotion_surprise,emotion_trust
0,0.0,0.0,0.0,0.000000,0.660312,0.0,0.750991,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.333333,0.000000,0.000000,0.000000
1,0.0,0.0,0.0,0.000000,0.660312,0.0,0.750991,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.500000,0.000000,0.000000,0.000000
2,0.0,0.0,0.0,0.000000,0.660312,0.0,0.750991,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.500000,0.000000,0.000000,0.000000
3,0.0,0.0,0.0,0.000000,0.660312,0.0,0.750991,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.500000,0.000000,0.000000,0.000000
4,0.0,0.0,0.0,0.000000,0.660312,0.0,0.750991,0.0,0.0,0.0,...,0.000000,0.000000,0.142857,0.000000,0.000000,0.142857,0.142857,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46482,0.0,0.0,0.0,0.707107,0.000000,0.0,0.000000,0.0,0.0,0.0,...,0.000000,0.142857,0.000000,0.000000,0.071429,0.071429,0.214286,0.071429,0.000000,0.071429
46483,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,...,0.100000,0.000000,0.200000,0.100000,0.000000,0.200000,0.100000,0.100000,0.000000,0.000000
46484,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,1.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.181818,0.000000,0.000000,0.090909,0.181818,0.090909,0.090909
46485,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,1.0,0.0,0.0,...,0.000000,0.066667,0.066667,0.000000,0.000000,0.000000,0.200000,0.000000,0.000000,0.000000


In [1]:
import ast
import pandas as pd
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import gc
import os
import pickle

def safe_parse_list(x):
    """안전하게 리스트를 파싱하는 함수"""
    try:
        if isinstance(x, str):
            return ast.literal_eval(x)
        elif isinstance(x, list):
            return x
        elif isinstance(x, np.ndarray):
            return x.tolist()
        elif hasattr(x, "tolist"):
            return x.tolist()
        else:
            return list(x) if x is not None else []
    except:
        return []

# ===== 1. 데이터 로딩 및 전처리 =====
print("데이터 로딩 중...")

# 케글 데이터 경로 설정
DATA_PATH = "/content/drive/MyDrive/LG 헬로비전 부트캠프/원본 데이터셋/데이터셋/two_tower"
user_df = pd.read_parquet(os.path.join(DATA_PATH, "user_sequence_df.parquet"))
item_df = pd.read_parquet(os.path.join(DATA_PATH, "item_df_tfidf_emotion.parquet"))

print(f"User 데이터 형태: {user_df.shape}")
print(f"Item 데이터 형태: {item_df.shape}")

# 데이터 샘플링 (메모리 절약을 위해)
SAMPLE_SIZE = 200000  # 케글 환경에 맞게 조정
if len(user_df) > SAMPLE_SIZE:
    print(f"데이터 샘플링: {len(user_df)} -> {SAMPLE_SIZE}")
    user_df = user_df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

# 리스트 데이터 안전하게 파싱
print("리스트 데이터 파싱 중...")
user_df["watched_item_ids"] = user_df["watched_item_ids"].apply(safe_parse_list)
user_df["watch_weights"] = user_df["watch_weights"].apply(safe_parse_list)
user_df["hour_seq"] = user_df["hour_seq"].apply(safe_parse_list)

# 최소 시청 이력이 있는 사용자만 필터링
min_seq_length = 3
user_df = user_df[user_df["watched_item_ids"].apply(len) >= min_seq_length].copy()

# 추가 샘플링 (메모리 효율성)
if len(user_df) > 100000:
    user_df = user_df.sample(n=100000, random_state=42).reset_index(drop=True)

print(f"필터링 후 사용자 수: {len(user_df)}")

# ===== 2. 인코딩 =====
print("인코딩 중...")

# 모든 아이템 ID 수집
all_item_ids = set(item_df["unique_asset_id"].astype(str).tolist())
print(f"아이템 데이터에서 수집된 ID 수: {len(all_item_ids)}")

# 사용자 시청 이력에서 아이템 ID 수집
user_item_ids = set()
for watched_list in tqdm(user_df["watched_item_ids"], desc="사용자 아이템 ID 수집"):
    user_item_ids.update([str(item) for item in watched_list])

print(f"사용자 시청 이력에서 수집된 ID 수: {len(user_item_ids)}")

# 교집합 계산 (실제 존재하는 아이템만)
valid_item_ids = all_item_ids.intersection(user_item_ids)
print(f"유효한 아이템 ID 수: {len(valid_item_ids)}")

# 인코더 생성
item_encoder = LabelEncoder()
item_encoder.fit(list(valid_item_ids))

user_encoder = LabelEncoder()
user_encoder.fit(user_df["user_id"])

# 사용자 인코딩
user_df["user_id_enc"] = user_encoder.transform(user_df["user_id"])

# 아이템 메타데이터 인코딩 (유효한 아이템만)
item_df["unique_asset_id_str"] = item_df["unique_asset_id"].astype(str)
item_df = item_df[item_df["unique_asset_id_str"].isin(valid_item_ids)].copy()
item_df["item_id_enc"] = item_encoder.transform(item_df["unique_asset_id_str"])

# 아이템 임베딩 매트릭스 생성
item_df_sorted = item_df.sort_values("item_id_enc")
item_features = item_df_sorted.drop(["unique_asset_id", "unique_asset_id_str", "item_id_enc"], axis=1).values
item_matrix = torch.tensor(item_features, dtype=torch.float32)

print(f"아이템 임베딩 매트릭스 크기: {item_matrix.shape}")

# 메모리 정리
del item_df_sorted, item_features
gc.collect()

# ===== 3. 학습 데이터 생성 (최적화된 버전) =====
def create_training_data_optimized(user_df, item_encoder, max_seq_length=30, max_samples_per_user=15):
    """최적화된 학습 데이터 생성"""
    training_data = []
    valid_classes = set(item_encoder.classes_)

    print("학습 데이터 생성 중...")
    for idx, row in tqdm(user_df.iterrows(), total=len(user_df), desc="Processing users"):
        user_id = row["user_id_enc"]
        watched_items = row["watched_item_ids"]

        # 유효한 아이템만 필터링
        valid_items = [str(item) for item in watched_items if str(item) in valid_classes]

        if len(valid_items) < 2:
            continue

        # 샘플 수 제한 (메모리 절약)
        max_samples = min(len(valid_items) - 1, max_samples_per_user)

        # 시퀀스에서 다음 아이템 예측 데이터 생성
        for i in range(1, min(len(valid_items), max_samples + 1)):
            history = valid_items[:i]
            target = valid_items[i]

            # 시퀀스 길이 제한
            if len(history) > max_seq_length:
                history = history[-max_seq_length:]

            try:
                training_data.append({
                    "user_id": user_id,
                    "history": item_encoder.transform(history).tolist(),
                    "target": item_encoder.transform([target])[0]
                })
            except ValueError:
                # 인코딩 실패 시 스킵
                continue

    return training_data

print("학습 데이터 생성 중...")
training_data = create_training_data_optimized(user_df, item_encoder, max_samples_per_user=8)
print(f"학습 샘플 수: {len(training_data)}")

# ===== 4. Dataset 클래스 =====
class TwoTowerDataset(Dataset):
    def __init__(self, data, max_seq_length=30):
        self.data = data
        self.max_seq_length = max_seq_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        history = sample["history"]

        # 패딩 또는 트렁케이션
        if len(history) > self.max_seq_length:
            history = history[-self.max_seq_length:]
        else:
            history = [0] * (self.max_seq_length - len(history)) + history

        return {
            "user_id": torch.tensor(sample["user_id"], dtype=torch.long),
            "history": torch.tensor(history, dtype=torch.long),
            "target": torch.tensor(sample["target"], dtype=torch.long)
        }

# ===== 5. Two-Tower with Attention 모델 =====
class TwoTowerWithAttention(nn.Module):
    def __init__(self, num_users, item_emb_matrix, embed_dim=256, num_heads=4, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim

        # User tower
        self.user_emb = nn.Embedding(num_users, embed_dim)

        # Item tower - 사전 훈련된 TF-IDF 임베딩 사용
        self.item_emb = nn.Embedding.from_pretrained(item_emb_matrix, freeze=False)

        # Item embedding을 embed_dim으로 맞추기 위한 projection
        if item_emb_matrix.shape[1] != embed_dim:
            self.item_proj = nn.Linear(item_emb_matrix.shape[1], embed_dim)
        else:
            self.item_proj = nn.Identity()

        # Self-attention for item sequence
        self.self_attention = nn.MultiheadAttention(
            embed_dim, num_heads=num_heads, dropout=dropout, batch_first=True
        )

        # Cross-attention between user and items
        self.cross_attention = nn.MultiheadAttention(
            embed_dim, num_heads=num_heads, dropout=dropout, batch_first=True
        )

        # Final projection layers
        self.user_proj = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, embed_dim)
        )

        self.layer_norm = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, user_id, item_history, attention_mask=None):
        batch_size = user_id.size(0)

        # User embedding
        user_emb = self.user_emb(user_id)  # (B, D)

        # Item sequence embedding
        item_emb = self.item_emb(item_history)  # (B, L, item_dim)
        item_emb = self.item_proj(item_emb)  # (B, L, D)

        # Create attention mask for padding
        if attention_mask is None:
            attention_mask = (item_history != 0).float()  # (B, L)

        # Self-attention on item sequence
        item_attended, _ = self.self_attention(item_emb, item_emb, item_emb)
        item_attended = self.layer_norm(item_attended + item_emb)

        # Aggregate item sequence (weighted average by attention mask)
        mask_expanded = attention_mask.unsqueeze(-1).expand_as(item_attended)
        item_sum = (item_attended * mask_expanded).sum(dim=1)
        item_count = attention_mask.sum(dim=1, keepdim=True).clamp(min=1.0)
        item_agg = item_sum / item_count  # (B, D)

        # Combine user and item representations
        user_expanded = user_emb.unsqueeze(1)  # (B, 1, D)
        item_agg_expanded = item_agg.unsqueeze(1)  # (B, 1, D)

        # Cross-attention
        user_attended, _ = self.cross_attention(
            user_expanded, item_agg_expanded, item_agg_expanded
        )
        user_attended = user_attended.squeeze(1)  # (B, D)

        # Final user representation
        user_final = self.user_proj(torch.cat([user_emb, user_attended], dim=-1))
        user_final = self.layer_norm(user_final)

        return user_final

    def get_item_embedding(self, item_ids):
        """아이템 임베딩 반환"""
        item_emb = self.item_emb(item_ids)
        return self.item_proj(item_emb)

# ===== 6. 학습 함수 =====
def train_epoch(model, train_loader, optimizer, device):
    model.train()
    total_loss = 0
    num_batches = 0

    progress_bar = tqdm(train_loader, desc="Training")
    for batch in progress_bar:
        user_id = batch["user_id"].to(device)
        history = batch["history"].to(device)
        target = batch["target"].to(device)

        # Attention mask 생성
        attention_mask = (history != 0).float()

        # Forward pass
        user_repr = model(user_id, history, attention_mask)

        # Negative sampling (간단한 in-batch negative sampling)
        batch_size = user_repr.size(0)

        # 타겟 아이템 임베딩
        target_emb = model.get_item_embedding(target)

        # 배치 내 다른 아이템들을 negative로 사용
        all_targets = target.unsqueeze(0).expand(batch_size, -1)  # (B, B)

        #all_target_emb = model.get_item_embedding(all_targets.view(-1)).view(batch_size, batch_size, -1)
        #all_target_emb = model.get_item_embedding(all_targets.reshape(-1)).reshape(batch_size, batch_size, -1)
        all_target_emb = model.get_item_embedding(all_targets.reshape(-1)).reshape(batch_size, batch_size, -1)

        # Dot product similarity
        scores = torch.bmm(user_repr.unsqueeze(1), all_target_emb.transpose(1, 2)).squeeze(1)  # (B, B)

        # Labels (대각선이 positive)
        labels = torch.arange(batch_size, device=device)

        # Cross entropy loss
        loss = F.cross_entropy(scores, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

        progress_bar.set_postfix({"Loss": f"{loss.item():.4f}"})

    return total_loss / num_batches

# ===== 7. 평가 함수 =====
def evaluate_model(model, val_loader, device, k=10):
    model.eval()
    total_hits = 0
    total_samples = 0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Evaluating"):
            user_id = batch["user_id"].to(device)
            history = batch["history"].to(device)
            target = batch["target"].to(device)

            attention_mask = (history != 0).float()
            user_repr = model(user_id, history, attention_mask)

            # 배치 내 아이템들에 대해서만 평가
            batch_size = user_repr.size(0)
            all_targets = target.unsqueeze(0).expand(batch_size, -1)
            #all_target_emb = model.get_item_embedding(all_targets.view(-1)).view(batch_size, batch_size, -1)
                    # ✅ reshape로 수정
            all_target_emb = model.get_item_embedding(all_targets.reshape(-1)).reshape(batch_size, batch_size, -1)


            scores = torch.bmm(user_repr.unsqueeze(1), all_target_emb.transpose(1, 2)).squeeze(1)

            # Top-k 예측
            _, top_k_indices = torch.topk(scores, k=min(k, batch_size), dim=1)

            # Hit rate 계산
            for i in range(batch_size):
                if i in top_k_indices[i]:
                    total_hits += 1
                total_samples += 1

    hit_rate = total_hits / total_samples if total_samples > 0 else 0
    return hit_rate

# ===== 8. 메인 실행 코드 =====
if __name__ == "__main__":
    # 하이퍼파라미터 (케글 환경에 맞게 조정)
    EMBED_DIM = 256
    NUM_HEADS = 4
    BATCH_SIZE = 32  # 케글 환경에 맞게 축소
    LEARNING_RATE = 1e-3
    NUM_EPOCHS = 5  # 케글 실행 시간 제한 고려
    MAX_SEQ_LENGTH = 30

    # 디바이스 설정
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"사용 디바이스: {device}")

    # 데이터셋 분할
    print("데이터셋 분할 중...")
    train_data, val_data = train_test_split(training_data, test_size=0.2, random_state=42)

    # DataLoader
    train_dataset = TwoTowerDataset(train_data, MAX_SEQ_LENGTH)
    val_dataset = TwoTowerDataset(val_data, MAX_SEQ_LENGTH)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    print(f"학습 배치 수: {len(train_loader)}")
    print(f"검증 배치 수: {len(val_loader)}")

    # 모델 초기화
    print("모델 초기화 중...")
    model = TwoTowerWithAttention(
        num_users=len(user_encoder.classes_),
        item_emb_matrix=item_matrix,
        embed_dim=EMBED_DIM,
        num_heads=NUM_HEADS
    ).to(device)

    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=1, factor=0.5)

    print(f"모델 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

    # 학습 루프
    best_hit_rate = 0
    for epoch in range(NUM_EPOCHS):
        print(f"\n=== Epoch {epoch + 1}/{NUM_EPOCHS} ===")

        # 학습
        train_loss = train_epoch(model, train_loader, optimizer, device)
        print(f"Train Loss: {train_loss:.4f}")

        # 평가
        hit_rate = evaluate_model(model, val_loader, device, k=10)
        print(f"Hit Rate@10: {hit_rate:.4f}")

        # 학습률 조정
        scheduler.step(hit_rate)

        # 베스트 모델 저장
        if hit_rate > best_hit_rate:
            best_hit_rate = hit_rate
            torch.save({
                'model_state_dict': model.state_dict(),
                'user_encoder': user_encoder,
                'item_encoder': item_encoder,
                'epoch': epoch,
                'hit_rate': hit_rate,
                'embed_dim': EMBED_DIM,
                'num_heads': NUM_HEADS
            }, '/content/drive/MyDrive/LG 헬로비전 부트캠프/원본 데이터셋/데이터셋/two_tower/best_two_tower_model.pth')
            print(f"새로운 베스트 모델 저장! Hit Rate: {hit_rate:.4f}")

        # 메모리 정리
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    # 추가 파일들 저장 (추천 시스템에서 사용)
    torch.save(item_matrix, '/content/drive/MyDrive/LG 헬로비전 부트캠프/원본 데이터셋/데이터셋/two_tower/item_matrix.pth')
    print("Item matrix 저장 완료!")

    # 인코더들도 저장
    with open('encoders.pkl', 'wb') as f:
        pickle.dump({
            'user_encoder': user_encoder,
            'item_encoder': item_encoder
        }, f)
    print("인코더 저장 완료!")

    print(f"\n학습 완료! 최고 Hit Rate@10: {best_hit_rate:.4f}")
    print("저장된 파일:")
    print("- best_two_tower_model.pth: 최고 성능 모델")
    print("- item_matrix.pth: 아이템 임베딩 매트릭스")
    print("- encoders.pkl: 사용자/아이템 인코더")

데이터 로딩 중...
User 데이터 형태: (424410, 5)
Item 데이터 형태: (46487, 36)
데이터 샘플링: 424410 -> 200000
리스트 데이터 파싱 중...
필터링 후 사용자 수: 100000
인코딩 중...
아이템 데이터에서 수집된 ID 수: 46487


사용자 아이템 ID 수집: 100%|██████████| 100000/100000 [00:00<00:00, 186550.40it/s]


사용자 시청 이력에서 수집된 ID 수: 26780
유효한 아이템 ID 수: 26274
아이템 임베딩 매트릭스 크기: torch.Size([26274, 35])
학습 데이터 생성 중...
학습 데이터 생성 중...


Processing users: 100%|██████████| 100000/100000 [8:23:29<00:00,  3.31it/s]


학습 샘플 수: 569169
사용 디바이스: cuda
데이터셋 분할 중...
학습 배치 수: 14230
검증 배치 수: 3558
모델 초기화 중...
모델 파라미터 수: 27,252,774

=== Epoch 1/5 ===


...

Hit Rate@10: 0.9294

=== Epoch 5/5 ===


Training: 100%|██████████| 14230/14230 [04:44<00:00, 49.96it/s, Loss=0.5521]


Train Loss: 1.2031


Evaluating: 100%|██████████| 3558/3558 [00:14<00:00, 248.55it/s]


Hit Rate@10: 0.9285
Item matrix 저장 완료!
인코더 저장 완료!

학습 완료! 최고 Hit Rate@10: 0.9304
저장된 파일:
- best_two_tower_model.pth: 최고 성능 모델
- item_matrix.pth: 아이템 임베딩 매트릭스
- encoders.pkl: 사용자/아이템 인코더


In [2]:
import ast
import pandas as pd
import torch
import numpy as np
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics.pairwise import cosine_similarity
import pickle
import os
from tqdm import tqdm

# TwoTowerWithAttention 클래스 재정의 (학습 코드와 동일)
class TwoTowerWithAttention(nn.Module):
    def __init__(self, num_users, item_emb_matrix, embed_dim=256, num_heads=4, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim

        # User tower
        self.user_emb = nn.Embedding(num_users, embed_dim)

        # Item tower - 사전 훈련된 TF-IDF 임베딩 사용
        self.item_emb = nn.Embedding.from_pretrained(item_emb_matrix, freeze=False)

        # Item embedding을 embed_dim으로 맞추기 위한 projection
        if item_emb_matrix.shape[1] != embed_dim:
            self.item_proj = nn.Linear(item_emb_matrix.shape[1], embed_dim)
        else:
            self.item_proj = nn.Identity()

        # Self-attention for item sequence
        self.self_attention = nn.MultiheadAttention(
            embed_dim, num_heads=num_heads, dropout=dropout, batch_first=True
        )

        # Cross-attention between user and items
        self.cross_attention = nn.MultiheadAttention(
            embed_dim, num_heads=num_heads, dropout=dropout, batch_first=True
        )

        # Final projection layers
        self.user_proj = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, embed_dim)
        )

        self.layer_norm = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, user_id, item_history, attention_mask=None):
        batch_size = user_id.size(0)

        # User embedding
        user_emb = self.user_emb(user_id)  # (B, D)

        # Item sequence embedding
        item_emb = self.item_emb(item_history)  # (B, L, item_dim)
        item_emb = self.item_proj(item_emb)  # (B, L, D)

        # Create attention mask for padding
        if attention_mask is None:
            attention_mask = (item_history != 0).float()  # (B, L)

        # Self-attention on item sequence
        item_attended, _ = self.self_attention(item_emb, item_emb, item_emb)
        item_attended = self.layer_norm(item_attended + item_emb)

        # Aggregate item sequence (weighted average by attention mask)
        mask_expanded = attention_mask.unsqueeze(-1).expand_as(item_attended)
        item_sum = (item_attended * mask_expanded).sum(dim=1)
        item_count = attention_mask.sum(dim=1, keepdim=True).clamp(min=1.0)
        item_agg = item_sum / item_count  # (B, D)

        # Combine user and item representations
        user_expanded = user_emb.unsqueeze(1)  # (B, 1, D)
        item_agg_expanded = item_agg.unsqueeze(1)  # (B, 1, D)

        # Cross-attention
        user_attended, _ = self.cross_attention(
            user_expanded, item_agg_expanded, item_agg_expanded
        )
        user_attended = user_attended.squeeze(1)  # (B, D)

        # Final user representation
        user_final = self.user_proj(torch.cat([user_emb, user_attended], dim=-1))
        user_final = self.layer_norm(user_final)

        return user_final

    def get_item_embedding(self, item_ids):
        """아이템 임베딩 반환"""
        item_emb = self.item_emb(item_ids)
        return self.item_proj(item_emb)

def safe_parse_list(x):
    """안전하게 리스트를 파싱하는 함수"""
    try:
        if isinstance(x, str):
            return ast.literal_eval(x)
        elif isinstance(x, list):
            return x
        elif isinstance(x, np.ndarray):
            return x.tolist()
        elif hasattr(x, "tolist"):
            return x.tolist()
        else:
            return list(x) if x is not None else []
    except:
        return []

class RecommendationSystem:
    def __init__(self, model_path='best_two_tower_model.pth',
                 item_matrix_path='item_matrix.pth',
                 encoders_path='encoders.pkl'):
        """추천 시스템 초기화"""
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # 모델 로드
        print("모델 로딩 중...")
        checkpoint = torch.load(model_path, map_location=self.device, weights_only=False)

        # 인코더 로드 (새로운 방식)
        if os.path.exists(encoders_path):
            with open(encoders_path, 'rb') as f:
                encoders = pickle.load(f)
            self.user_encoder = encoders['user_encoder']
            self.item_encoder = encoders['item_encoder']
        else:
            # 체크포인트에서 로드 (이전 방식)
            self.user_encoder = checkpoint['user_encoder']
            self.item_encoder = checkpoint['item_encoder']

        # 아이템 매트릭스 로드
        item_matrix = torch.load(item_matrix_path, map_location=self.device)

        # 모델 재구성
        embed_dim = checkpoint.get('embed_dim', 256)
        num_heads = checkpoint.get('num_heads', 4)

        self.model = TwoTowerWithAttention(
            num_users=len(self.user_encoder.classes_),
            item_emb_matrix=item_matrix,
            embed_dim=embed_dim,
            num_heads=num_heads
        ).to(self.device)

        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model.eval()

        print(f"모델 로드 완료. 사용자 수: {len(self.user_encoder.classes_)}, 아이템 수: {len(self.item_encoder.classes_)}")

    def extract_user_vectors(self, user_df, batch_size=64):
        """모든 사용자의 레이턴트 벡터 추출"""
        print("사용자 레이턴트 벡터 추출 중...")

        user_vectors = {}

        for i in tqdm(range(0, len(user_df), batch_size), desc="Extracting user vectors"):
            batch_df = user_df.iloc[i:i+batch_size]

            user_ids = []
            histories = []
            attention_masks = []
            original_user_ids = []

            for _, row in batch_df.iterrows():
                if row["user_id"] not in self.user_encoder.classes_:
                    continue

                user_id_enc = self.user_encoder.transform([row["user_id"]])[0]
                watched_items = safe_parse_list(row["watched_item_ids"])
                valid_items = [str(item) for item in watched_items if str(item) in self.item_encoder.classes_]

                if not valid_items:
                    continue

                history = self.item_encoder.transform(valid_items).tolist()
                max_len = 30
                if len(history) > max_len:
                    history = history[-max_len:]
                else:
                    history = [0] * (max_len - len(history)) + history

                user_ids.append(user_id_enc)
                histories.append(history)
                attention_masks.append([1 if x != 0 else 0 for x in history])
                original_user_ids.append(row["user_id"])

            if not user_ids:
                continue

            user_ids_tensor = torch.tensor(user_ids, dtype=torch.long).to(self.device)
            histories_tensor = torch.tensor(histories, dtype=torch.long).to(self.device)
            attention_masks_tensor = torch.tensor(attention_masks, dtype=torch.float32).to(self.device)

            with torch.no_grad():
                user_vec = self.model(user_ids_tensor, histories_tensor, attention_masks_tensor)
                user_vec = user_vec.cpu().numpy()

            for uid, vec in zip(original_user_ids, user_vec):
                user_vectors[uid] = vec

        return user_vectors

In [3]:
# 예시 실행 코드
from pathlib import Path

# 유저 시퀀스 로딩 (학습에 썼던 거 재사용 가능)
user_df = pd.read_parquet(Path("/content/drive/MyDrive/LG 헬로비전 부트캠프/원본 데이터셋/데이터셋/two_tower/user_sequence_df.parquet"))
user_df["watched_item_ids"] = user_df["watched_item_ids"].apply(safe_parse_list)

# 샘플링 (선택)
user_df = user_df.sample(n=1000, random_state=42).reset_index(drop=True)

# 시스템 초기화 및 벡터 추출
system = RecommendationSystem(
    model_path="/content/drive/MyDrive/LG 헬로비전 부트캠프/원본 데이터셋/데이터셋/two_tower/best_two_tower_model.pth",
    item_matrix_path="/content/drive/MyDrive/LG 헬로비전 부트캠프/원본 데이터셋/데이터셋/two_tower/item_matrix.pth",
    encoders_path="/content/drive/MyDrive/LG 헬로비전 부트캠프/원본 데이터셋/데이터셋/two_tower/encoders.pkl"
)

user_vectors = system.extract_user_vectors(user_df)

# 저장
np.save("user_latent_vectors.npy", user_vectors)
print("✅ 유저 레이턴트 벡터 추출 및 저장 완료")


모델 로딩 중...
모델 로드 완료. 사용자 수: 100000, 아이템 수: 26274
사용자 레이턴트 벡터 추출 중...


Extracting user vectors: 100%|██████████| 16/16 [00:16<00:00,  1.04s/it]

✅ 유저 레이턴트 벡터 추출 및 저장 완료


In [4]:
def recommend_top_k(user_vectors, item_matrix, item_encoder, k=10):
    """모든 사용자에 대해 Top-k 추천 결과 생성"""
    item_vecs = item_matrix.cpu().numpy()
    results = []

    for user_id, user_vec in user_vectors.items():
        sims = cosine_similarity([user_vec], item_vecs)[0]
        top_k_idx = sims.argsort()[::-1][:k]

        for rank, idx in enumerate(top_k_idx):
            asset_id = item_encoder.classes_[idx]
            score = sims[idx]
            results.append({
                'user_id': user_id,
                'asset_id': asset_id,
                'rank': rank + 1,
                'score': float(score)
            })

    return pd.DataFrame(results)



In [5]:
# 1. 유저 벡터 추출
user_vectors = system.extract_user_vectors(user_df)

# 2. 아이템 임베딩 전부 추출
item_ids = torch.arange(len(system.item_encoder.classes_)).to(system.device)
item_matrix = system.model.get_item_embedding(item_ids).detach()

# 3. 추천 리스트 생성
recommend_df = recommend_top_k(user_vectors, item_matrix, system.item_encoder, k=10)

# 4. 저장
recommend_df.to_csv("/content/drive/MyDrive/LG 헬로비전 부트캠프/원본 데이터셋/데이터셋/two_tower/top10_recommendations.csv", index=False)
print("✅ 추천 리스트 저장 완료")


사용자 레이턴트 벡터 추출 중...


Extracting user vectors: 100%|██████████| 16/16 [00:15<00:00,  1.01it/s]


✅ 추천 리스트 저장 완료
